# Data Preparation of Competitor Documents

The C-SEO Bench dataset (Puerto et al., 2025) contains 500 queries, each with exactly 10 documents. For each query, one document is designated as the target — these selections are defined by Puerto et al. and stored in `dataset/selected_docs.json`. This notebook extracts only the **competitor documents** (the 9 non-target documents per query) and saves them for use in the retrieval stage.

**Input:**
- HuggingFace (`parameterlab/c-seo-bench`, retail split) — 500 queries × 10 documents
- `data/retail/dataset/selected_docs.json` — target document index per query, downloaded from the [C-SEO Bench GitHub repository](https://github.com/parameterlab/c-seo-bench/blob/main/data/retail/selected_docs.json)

**Output:**
- `data/retail/dataset/1_preparation_competitor_docs_v1.json` — the 9 competitor documents per query

## Structure
1. **Setup** — imports and paths
2. **Load Dataset** — downloads retail split from HuggingFace
3. **Save Competitor Docs** — extracts and saves all non-target documents per query
4. **Inspect** — verifies output for a sample query

## 1. Setup

Imports required libraries and defines file paths relative to the project root.

In [ ]:
import json
import os

from datasets import load_dataset

# Determine project root relative to this notebook's location
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, ".."))

# Data directory for all retail experiment files
data_dir = os.path.join(project_root, "data", "retail")
os.makedirs(data_dir, exist_ok=True)

# Input: selected_docs.json from Puerto et al. — target document index per query (read-only)
selected_docs_path   = os.path.join(data_dir, "dataset", "selected_docs.json")

# Output: competitor documents per query
competitor_docs_path = os.path.join(data_dir, "dataset", "1_preparation_competitor_docs_v1.json")

print("Setup complete.")
print(f"Project root: {project_root}")
print(f"Output file:  {competitor_docs_path}")


## 2. Load Dataset

Downloads the retail split of the C-SEO Bench dataset from HuggingFace (`parameterlab/c-seo-bench`) and converts it to a DataFrame.

In [ ]:
# Load the retail split of the C-SEO Bench dataset from HuggingFace
# Dataset: https://huggingface.co/datasets/parameterlab/c-seo-bench
print("Loading retail dataset from HuggingFace...")
ds = load_dataset("parameterlab/c-seo-bench", split="retail")

# Convert to pandas DataFrame for easier manipulation
df = ds.to_pandas()

# Verify the dataset loaded correctly
print(f"Loaded {len(df)} rows")
print(f"Columns: {list(df.columns)}")
print(f"Unique queries: {df['query_id'].nunique()}")
print()
print("First row:")
print(df.iloc[0])

## 3. Save Competitor Docs

For each of the 500 queries, reads the target document index from `selected_docs.json` and saves all remaining documents as competitor docs to `1_preparation_competitor_docs_v1.json`. Does not modify `selected_docs.json`.

In [ ]:
# Load selected_docs.json to retrieve the target document index per query
with open(selected_docs_path, "r", encoding="utf-8") as f:
    selected_docs = json.load(f)

# Get all unique query IDs from the dataset in order
query_ids = df["query_id"].unique()
competitor_docs = {}

for query_idx, query_id in enumerate(query_ids):
    query_idx_str = str(query_idx)

    # Skip queries that are not in selected_docs — should not happen but safety check
    if query_idx_str not in selected_docs:
        print(f"[{query_idx}] WARNING: not in selected_docs — skipping")
        continue

    # Extract the target document index — first key in the selected_docs entry
    target_doc_idx = int(list(selected_docs[query_idx_str].keys())[0])

    # Get all 10 documents for this query from the dataset
    hits = df[df["query_id"] == query_id].reset_index(drop=True)
    query_text = hits["query"].iloc[0]

    # Collect all non-target documents as competitor docs
    competitors = []
    for doc_idx, row in hits.iterrows():
        if doc_idx == target_doc_idx:
            continue  # skip target doc
        competitors.append({
            "doc_idx": doc_idx,
            "doc": row["document"],
        })
    
    # Store the full entry for this query
    competitor_docs[query_idx_str] = {
        "query_id":        int(query_id),
        "query":           query_text,
        "target_doc_idx":  target_doc_idx,
        "competitor_docs": competitors,
    }

# Save 1_preparation_competitor_docs_v1.json — does NOT modify selected_docs.json
with open(competitor_docs_path, "w", encoding="utf-8") as f:
    json.dump(competitor_docs, f, indent=4, ensure_ascii=False)

print(f"Saved 1_preparation_competitor_docs_v1.json — {len(competitor_docs)} queries")
print(f"Example: query 0 has {len(competitor_docs['0']['competitor_docs'])} competitor docs")

## 4. Inspect

Reloads the saved output file and prints a summary for a sample query. Change `INSPECT_QUERY_IDX` to verify any entry (0–499).

In [ ]:
# Reload the saved file to verify the output is correct
with open(competitor_docs_path, "r", encoding="utf-8") as f:
    competitor_docs = json.load(f)

# Change this index to inspect a different query (0 to 499)
INSPECT_QUERY_IDX = "0"
example = competitor_docs[INSPECT_QUERY_IDX]

# Print summary for the selected query
print(f"Query idx:       {INSPECT_QUERY_IDX}")
print(f"Query ID:        {example['query_id']}")
print(f"Query:           {example['query']}")
print(f"Target doc idx:  {example['target_doc_idx']}")
print(f"Competitor docs: {len(example['competitor_docs'])}")
print()

# Preview the first competitor document — truncated to 200 characters
print("First competitor doc (first 200 chars):")
print(example['competitor_docs'][0]['doc'][:200])